# しょぼんｎの段位期別分布と平滑化
S1 → S2 → S3 の順に実行します。C5の完了キャッシュと元の実行フォルダを読みます。旧Notebookの再実行、ZIPの読み込み、モデル学習は不要です。

1・10・20対局で、段位群内のばらつきと群間の差・重なりを比較します。10・20対局は時間帯の途中で切らず、同じ段位群の連続した観測対局だけを使います。段位17・初期除外・共通対局をまたぐ窓は作りません。

In [ ]:
# S1　Drive接続・固定コードの取得・合成データによる動作確認
from google.colab import drive
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import tempfile
import urllib.request

drive.mount('/content/drive')
COMMIT = 'e7a2c414afa40f74ca8de2bbb2a8daffbd6a8dae'
source_root = Path(tempfile.mkdtemp(prefix='mahjong_smoothing_'))
code_dir = source_root / 'smoothing'
base = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/'
for name in ('mahjong_research/__init__.py', 'mahjong_research/tenhou.py',
             'mahjong_research/analysis.py', 'mahjong_research/inputs.py', 'mahjong_research/legacy.py',
             'smoothing/study.py', 'smoothing/presentation.py', 'smoothing/test_smoothing.py'):
    dst = source_root / name
    dst.parent.mkdir(exist_ok=True)
    with urllib.request.urlopen(base + name, timeout=60) as response:
        dst.write_bytes(response.read())
deps = {'pandas':'pandas', 'numpy':'numpy', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [pkg for module, pkg in deps.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
env = dict(os.environ, PYTHONPATH=str(source_root), MPLBACKEND='Agg')
subprocess.run([sys.executable, '-m', 'unittest', '-q', 'test_smoothing'], cwd=code_dir, env=env, check=True)
print('合成データの動作確認が完了しました。')

In [ ]:
# S2　過去のC3・C5で使った実行フォルダを選択（候補が1件なら自動）
ROOT = Path('/content/drive/MyDrive/卒論')
RUN_DIR = None  # 例: ROOT / 'research_v2_runs' / '実行フォルダ名'
OUTPUT_ROOT = ROOT / 'research_v2_smoothing'
if RUN_DIR is None:
    choices = sorted(p.parent for p in (ROOT/'research_v2_runs').glob('*/input_plan.json')
                     if (p.parent/'shared_games.csv').is_file())
    if len(choices) != 1:
        print(*choices, sep='\n')
        raise ValueError('ROOTを確認し、C5が完了した実行フォルダをRUN_DIRに指定してS2を再実行してください。')
    RUN_DIR = choices[0]
RUN_DIR = Path(RUN_DIR)
print('既存の入力:', RUN_DIR, '\n新しい出力の親:', OUTPUT_ROOT)

In [ ]:
# S3　キャッシュ監査・両段位期の比較・CSV/図/日本語レポート保存
from IPython.display import Markdown, display
import pandas as pd
runner = 'import sys; from study import run_study; print(run_study(sys.argv[1], sys.argv[2]))'
# 別プロセスで固定コードを読み込み、既存C1〜C10の読み込み状態と分離する
completed = subprocess.run([sys.executable, '-c', runner, str(RUN_DIR), str(OUTPUT_ROOT)],
                           cwd=code_dir, env=env, stdout=subprocess.PIPE, text=True, check=True)
RESULT = Path(completed.stdout.strip().splitlines()[-1])
print('保存先:', RESULT)
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf-8')))
display(pd.read_csv(RESULT/'feature_candidates.csv')[['width','japanese','category','diff','smd','overlap','auc',
                                                      'low_valid_n','high_valid_n','common_smd']])

**成果物の読み方**

- `comparison_20x3.csv` は各幅で採用可能な全単位、`common_games_20x3.csv` は同じ元対局に限定した比較です。差の符号は高段位期−低段位期です。
- `group_statistics.csv` は両対象集合の群別分布、`window_membership.csv` は単位別の元対局IDです。同一時間帯内のIDの記載順に時系列としての意味はありません。
- `feature_candidates.csv` は探索上のA〜D、判断根拠、戦術的な意味、共通対局の結果を含みます。`figures/` に全20項目の箱ひげ図・SD・SMD・欠損率・機会数・散布図があります。
- 再実行ごとに新規フォルダを作成します。監査エラー時は参照先とC5の完了記録を確認してください。入力が不足する場合はエラーに表示されたファイルや列が必要です。